# ARIMA(10,1,10) Market Cap Rank Prediction — All Rebalance Years (2006–2025)

---

## Overview

This is the **second baseline notebook** in the predictions series. It predicts Russell 2000 market cap ranks 60 trading days after each annual rank date using a **fixed ARIMA(10, 1, 10)** model — the same order for every stock, every year.

The study covers all 20 Russell annual reconstitutions (2006–2025). All evaluation metrics match notebooks 02 (AR10) and 04 (Factor-ARIMA) for direct comparison.

---

## Why ARIMA(10, 1, 10) as a Baseline?

Notebook 02 uses AR(10) — a pure autoregressive model on first-differenced market caps. This notebook adds a **moving-average component of order 10**, making the model ARMA(10,10) on first-differences, i.e., ARIMA(10, 1, 10).

The MA(q) term allows the model to directly account for the effect of past shocks on today's market cap change:

$$\Delta M_t = c + \sum_{k=1}^{10} \phi_k \, \Delta M_{t-k} + \varepsilon_t + \sum_{j=1}^{10} \theta_j \, \varepsilon_{t-j}$$

For small-cap equities that mean-revert after large single-day moves, the MA component captures this reversion in a way pure AR cannot.

**Why fixed orders, not BIC-selected?**  
A BIC grid search over (p, q) adds substantial computation without reliably improving 60-day forecasts — the dominant signal at this horizon is "large-cap stocks stay large," a pattern captured equally well by any reasonable fixed-order ARIMA. Using fixed orders keeps the model transparent, the notebook fast, and the comparison clean.

**d = 1 fixed:** Market cap levels are non-stationary (continuously growing or shrinking firms), so first-differencing is appropriate for all stocks without needing a per-stock ADF test.

---

## Comparison to Prior Notebooks

| | Notebook 02 (AR10) | **This notebook** | Notebook 04 (Factor-ARIMA) |
|---|---|---|---|
| Stock model | AR(10) on ΔMCAP | **ARIMA(10,1,10) on ΔMCAP** | OLS factor loadings |
| Factor model | None | None | ARIMA(5, d_adf, 5) per factor |
| d selection | Fixed d=1 | **Fixed d=1** | ADF per factor |
| Order selection | Fixed | **Fixed** | Fixed |
| Forecast horizon | 60 days | 60 days | 60 days |

---

## 1. Setup

All imports, paths, and model parameters are defined here. Change `MAX_P`, `MAX_Q`, or `STEPS` to experiment with different configurations.

In [ ]:
import warnings
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from statsmodels.tsa.arima.model import ARIMA
from joblib import Parallel, delayed

sns.set_theme(style='whitegrid', font_scale=1.05)
warnings.filterwarnings('ignore')

# ── Data paths ────────────────────────────────────────────────────────────────
PROCESSED_DIR  = Path('../../data/processed')
RAW_PRICES_DIR = Path('../../data/raw/prices')
RAW_MBR_DIR    = Path('../../data/raw/membership')

MCAPS_CSV      = RAW_PRICES_DIR / 'FINALIZED_MCAPS_CSV.csv'
MCAPS_PARQUET  = PROCESSED_DIR  / 'FINALIZED_MCAPS.parquet'
CALENDAR_FILE  = RAW_MBR_DIR    / 'russell_calendar.xlsx'
CACHE_DIR      = PROCESSED_DIR  / 'arima_1010_cache'
CACHE_DIR.mkdir(exist_ok=True)

# ── Model parameters ─────────────────────────────────────────────────────────
AR_ORDER   = 10    # fixed AR order
MA_ORDER   = 10    # fixed MA order
D_ORDER    = 1     # fixed differencing order
STEPS      = 60    # forecast horizon in trading days (~3 calendar months)
TRAIN_DAYS = 315   # max training window (≈15 months of trading days)
MIN_OBS    = 40    # minimum non-NaN observations required to fit
N_JOBS     = -1    # use all available CPU cores

print(f'Model              : ARIMA({AR_ORDER}, {D_ORDER}, {MA_ORDER})  — fixed orders, no search')
print(f'Forecast horizon   : {STEPS} trading days')
print(f'Training window    : up to {TRAIN_DAYS} trading days')
print(f'Min observations   : {MIN_OBS}')

---

## 2. Data Loading

### 2.1 Market Cap Matrix

The source file `FINALIZED_MCAPS_CSV.csv` contains daily market caps (USD millions) for ~11,500 Bloomberg tickers from January 2006 to mid-2026. The cell below converts it to Parquet on first run (~30 seconds) and loads it instantly on subsequent runs.

**Bloomberg quirks handled:**
- UTF-8 BOM in the header (`encoding='utf-8-sig'`)
- Missing values encoded as `#N/A N/A` rather than blank
- Date index stored as M/D/YYYY strings → converted to `datetime`

In [7]:
if not MCAPS_PARQUET.exists():
    print('Converting MCAPS CSV → Parquet (one-time, ~30s)...')
    _m = pd.read_csv(
        MCAPS_CSV,
        index_col=0,
        na_values=['#N/A N/A', '#N/A', 'N/A', '#NA'],
        encoding='utf-8-sig'
    )
    _m.index = pd.to_datetime(_m.index)
    _m = _m.apply(pd.to_numeric, errors='coerce')
    _m.sort_index().to_parquet(MCAPS_PARQUET)
    print(f'Done. Shape: {_m.shape}')
else:
    print('Parquet already exists — skipping conversion.')

mcaps = pd.read_parquet(MCAPS_PARQUET)
print(f'\nMarket cap matrix : {mcaps.shape[0]:,} dates × {mcaps.shape[1]:,} tickers')
print(f'Date range        : {mcaps.index.min().date()} → {mcaps.index.max().date()}')
print(f'Coverage          : {mcaps.notna().mean().mean():.1%} of cells non-NaN')

Parquet already exists — skipping conversion.

Market cap matrix : 5,310 dates × 11,545 tickers
Date range        : 2006-01-02 → 2026-05-08
Coverage          : 79.0% of cells non-NaN


### 2.2 Russell Rebalance Calendar

The Rank Date for each year is `t = 0` — training data runs up to and including this date; the forecast target is the 60th trading day after it (~early August each year).

Key dates in the annual cycle:

| Milestone | Typical timing | Role in this study |
|---|---|---|
| **Rank Date** | Late April / May | `t = 0` — end of training window |
| Announcement Date | ~23 trading days after rank | Preliminary adds/deletes published |
| Effective Date | Last Friday of June (~38 trading days after rank) | Index reconstitutes |
| **Forecast target** | 60 trading days after rank | ~early August — evaluation point |

In [8]:
_cal_raw = pd.read_excel(CALENDAR_FILE, sheet_name='Russell Calendar', header=3)
_cal_raw.columns = ['Year', 'Period', 'Rank_Date', 'Announcement_Date',
                    'Effective_Date', 'Effective_Open', 'Notes']
calendar = (
    _cal_raw[['Year', 'Rank_Date', 'Effective_Date']]
    .dropna(subset=['Year'])
    .assign(
        Year           = lambda d: d['Year'].astype(int),
        Rank_Date      = lambda d: pd.to_datetime(d['Rank_Date']),
        Effective_Date = lambda d: pd.to_datetime(d['Effective_Date']),
    )
    .set_index('Year')
)
print(f'Calendar: {len(calendar)} years  ({calendar.index.min()} – {calendar.index.max()})')
calendar

Calendar: 20 years  (2006 – 2025)


,Rank_Date,Effective_Date
Year,,
2006,2006-05-31,2006-06-30
2007,2007-05-31,2007-06-22
2008,2008-05-30,2008-06-27
2009,2009-05-29,2009-06-26
2010,2010-05-28,2010-06-25
2011,2011-05-31,2011-06-24
2012,2012-05-31,2012-06-22
2013,2013-05-31,2013-06-28
2014,2014-05-30,2014-06-27


---

## 3. Model Definition

### 3.1 ARIMA(10, 1, 10) — Fixed Orders

For each stock, we fit a single ARIMA(10, 1, 10) model on its daily market cap series:

- **AR order p = 10**: the change in market cap today depends on the 10 most recent changes
- **Differencing d = 1**: we model first-differences ΔM_t rather than levels
- **MA order q = 10**: shocks from the 10 most recent days feed directly into today's change

No ADF unit root test, no model selection — just fit ARIMA(10, 1, 10) to the training window and project 60 steps forward. This is deliberately simple: if a sophisticated model cannot beat this baseline, it suggests the forecast task is fundamentally limited by unpredictability of relative market cap moves, not by model complexity.

### 3.2 Parallelisation and Caching

Each stock is fit independently (embarrassingly parallel). `joblib.Parallel(prefer='threads')` uses all CPU cores. Results are cached per year to `data/processed/arima_1010_cache/` so interrupted runs resume without repeating work.

### 3.3 Russell Index Membership Bands

Stylised rank-based proxies used consistently throughout this project.

In [ ]:
def fit_arima(ticker, series, steps):
    """
    Fit ARIMA(10, 1, 10) on one stock's market cap series and return
    a 60-day-ahead point forecast.

    Returns
    -------
    (ticker, predicted_mktcap)
    predicted_mktcap is None if fitting fails or series is too short.
    """
    s = series.dropna()
    if len(s) < MIN_OBS:
        return (ticker, None)

    try:
        res         = ARIMA(s, order=(AR_ORDER, D_ORDER, MA_ORDER)).fit()
        pred_mktcap = float(res.get_forecast(steps=steps).predicted_mean.iloc[-1])
        return (ticker, pred_mktcap)
    except Exception:
        return (ticker, None)


RUSSELL_BANDS = {
    'Russell 1000':     (1,    1000),
    'Russell 2000':     (1001, 3000),
    'Russell 3000':     (1,    3000),
    'Russell Microcap': (2001, 4000),
}

def in_band(rank_col, lo, hi):
    return (rank_col >= lo) & (rank_col <= hi)

---

## 4. Main Loop — Run Study for Every Rebalance Year

For each year the loop:

1. **Checks the cache** — if a pickle exists for this year, loads it and moves on. Safe to re-run after interruption.
2. **Slices training data** — up to 315 trading days ending on the rank date.
3. **Dispatches parallel ARIMA fits** — one ARIMA(10, 1, 10) per eligible stock using all CPU cores.
4. **Ranks and evaluates** — predicted vs actual market caps at the 60-day target date.
5. **Attaches prior-year snapshot** — for add/remove analysis (prior year's rank date market caps used as pre-rebalance membership proxy).
6. **Saves to cache** and prints a one-line summary.

In [ ]:
year_results = {}

for year, row in calendar.iterrows():
    rank_date  = row['Rank_Date']
    cache_file = CACHE_DIR / f'year_{year}.pkl'

    # ── Load from cache if available ──────────────────────────────────────────
    if cache_file.exists():
        with open(cache_file, 'rb') as f:
            cached = pickle.load(f)
        year_results[year] = cached['cmp']
        rho, _ = spearmanr(cached['cmp']['predicted'], cached['cmp']['actual'])
        mae    = cached['cmp']['rank_error'].mean()
        print(f'[{year}] loaded from cache  n={len(cached["cmp"]):,}  ρ={rho:.3f}  MAE={mae:.0f}')
        continue

    # ── Slice training data ───────────────────────────────────────────────────
    all_before = mcaps[mcaps.index <= rank_date]
    train      = all_before.iloc[-TRAIN_DAYS:]
    all_after  = mcaps[mcaps.index > rank_date]

    if len(all_after) < STEPS:
        print(f'[{year}] insufficient post-rank data — skipping')
        continue

    eligible = [tkr for tkr in train.columns
                if train[tkr].dropna().shape[0] >= MIN_OBS]

    # ── Parallel ARIMA(10,1,10) fits per stock ────────────────────────────────
    fit_results = Parallel(n_jobs=N_JOBS, prefer='threads')(
        delayed(fit_arima)(tkr, train[tkr], STEPS)
        for tkr in eligible
    )

    # ── Assemble predictions ──────────────────────────────────────────────────
    predictions = {}
    for tkr, pred in fit_results:
        if pred is not None:
            predictions[tkr] = pred

    # ── Actual market cap at the 60-day target date ───────────────────────────
    actual_at_target = all_after.iloc[STEPS - 1]
    target_date      = all_after.index[STEPS - 1]

    pred_s   = pd.Series(predictions)
    actual_s = actual_at_target.reindex(pred_s.index)
    cmp      = pd.DataFrame({'predicted': pred_s, 'actual': actual_s}).dropna()

    if len(cmp) < 100:
        print(f'[{year}] too few stocks ({len(cmp)}) — skipping')
        continue

    cmp['pred_rank']   = cmp['predicted'].rank(ascending=False)
    cmp['actual_rank'] = cmp['actual'].rank(ascending=False)
    cmp['rank_error']  = (cmp['pred_rank'] - cmp['actual_rank']).abs()

    # Prior-year rank date snapshot → pre-rebalance membership proxy
    prior_rank_date = calendar.loc[year - 1, 'Rank_Date'] if (year - 1) in calendar.index else None
    if prior_rank_date is not None:
        prior_snap       = mcaps[mcaps.index <= prior_rank_date].iloc[-1]
        cmp['prev_rank'] = prior_snap.reindex(cmp.index).rank(ascending=False)
    else:
        cmp['prev_rank'] = np.nan

    cmp['year']        = year
    cmp['target_date'] = target_date
    cmp['n_train']     = len(train)

    # ── Cache to disk ─────────────────────────────────────────────────────────
    with open(cache_file, 'wb') as f:
        pickle.dump({'cmp': cmp}, f)

    year_results[year] = cmp

    rho, _ = spearmanr(cmp['predicted'], cmp['actual'])
    mae    = cmp['rank_error'].mean()
    print(f'[{year}]  rank_date={rank_date.date()}  target={target_date.date()}  '
          f'n={len(cmp):,}  ρ={rho:.3f}  MAE={mae:.0f}')

print(f'\nCompleted {len(year_results)} / {len(calendar)} years.')

---

## 5. Results

### 5.1 Per-Year Summary Table

Layout matches the AR(10) notebook exactly for side-by-side comparison:

- **Train Days**: actual observations in the training window (may be <315 in early years)
- **N Stocks**: stocks with a valid forecast and actual market cap at the target date
- **Spearman ρ**: rank correlation between predicted and actual market caps
- **MAE Rank**: mean absolute displacement in rank position
- **Median Err**: median rank error (more robust to outliers than the mean)

In [ ]:
rows = []
for year, cmp in year_results.items():
    rho, pval = spearmanr(cmp['predicted'], cmp['actual'])
    rows.append({
        'Year':        year,
        'Rank Date':   calendar.loc[year, 'Rank_Date'].date(),
        'Target Date': cmp['target_date'].iloc[0].date(),
        'Train Days':  cmp['n_train'].iloc[0],
        'N Stocks':    len(cmp),
        'Spearman ρ':  round(rho, 4),
        'p-value':     round(pval, 4),
        'MAE Rank':    round(cmp['rank_error'].mean(), 1),
        'Median Err':  round(cmp['rank_error'].median(), 1),
    })

summary = pd.DataFrame(rows).set_index('Year')
summary

### 5.3 Spearman ρ and MAE Rank by Year

These two charts mirror the AR(10) notebook's layout exactly for direct visual comparison.

**Interpretation guidance:**
- **High Spearman ρ** is expected regardless of model quality — large-cap stocks (Apple, Microsoft, etc.) are consistently ranked near the top, which mechanically boosts the rank correlation. The relevant signal is the *delta* between Auto-ARIMA and AR(10) ρ values.
- **MAE Rank** is the more honest metric. With ~3,500 stocks, an MAE of 300 means the average stock is off by ~8.5% of the total rank range. Crisis years (2008, 2020) typically show the largest errors because extreme market cap moves are precisely what a trend-extrapolation model cannot anticipate.

In [ ]:
years = summary.index.tolist()
rhos  = summary['Spearman ρ'].tolist()
maes  = summary['MAE Rank'].tolist()

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

ax = axes[0]
ax.bar(years, rhos, color='steelblue', edgecolor='white', zorder=3)
ax.axhline(0,             color='black',  lw=0.8, zorder=2)
ax.axhline(1,             color='green',  lw=0.8, linestyle='--', label='Perfect (ρ = 1)', zorder=2)
ax.axhline(np.mean(rhos), color='orange', lw=1.2, linestyle=':',
           label=f'Mean ρ = {np.mean(rhos):.3f}', zorder=2)
ax.set_ylabel('Spearman ρ', fontsize=11)
ax.set_title('ARIMA(10,1,10) — Spearman ρ by Year  (60-day horizon)', fontsize=12)
ax.set_ylim(-0.05, 1.08)
ax.legend(fontsize=9)
for x, v in zip(years, rhos):
    ax.text(x, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=7.5)

ax = axes[1]
ax.bar(years, maes, color='darkorange', edgecolor='white', zorder=3)
ax.axhline(np.mean(maes), color='steelblue', lw=1.2, linestyle=':',
           label=f'Mean MAE = {np.mean(maes):.0f}', zorder=2)
ax.set_ylabel('Mean Absolute Rank Error', fontsize=11)
ax.set_xlabel('Rebalance Year', fontsize=11)
ax.set_title('Mean Absolute Rank Error by Year', fontsize=12)
ax.legend(fontsize=9)
for x, v in zip(years, maes):
    ax.text(x, v + 1, f'{v:.0f}', ha='center', va='bottom', fontsize=7.5)

plt.tight_layout()
plt.show()

### 5.4 Russell Index Membership Overlap

For each Russell index and each year: of the stocks that *actually* ended up in the index at the 60-day target date, what fraction did our predicted ranks also place there?

$$\text{Overlap}\% = \frac{|\text{Predicted members} \cap \text{Actual members}|}{|\text{Actual members}|} \times 100$$

**Expected pattern by index:**
- **R1000**: high overlap (85–95%) — mega-cap stocks rarely fall below rank 1,000 in 60 days
- **R2000**: moderate (70–85%) — the small-cap boundary is more contested; 60-day moves can push borderline stocks in or out
- **Microcap**: lowest — most volatile segment; the model has least signal here

The orange dotted line shows the cross-year mean for each index.

In [ ]:
overlap_rows = []
for year, cmp in year_results.items():
    for name, (lo, hi) in RUSSELL_BANDS.items():
        pred_in   = set(cmp[in_band(cmp['pred_rank'],   lo, hi)].index)
        actual_in = set(cmp[in_band(cmp['actual_rank'], lo, hi)].index)
        n_actual  = len(actual_in)
        correct   = len(pred_in & actual_in)
        overlap_rows.append({
            'Year':        year,
            'Index':       name,
            'N_Actual':    n_actual,
            'Overlap_Pct': correct / n_actual * 100 if n_actual > 0 else 0,
        })

overlap_df = pd.DataFrame(overlap_rows)

fig, axes = plt.subplots(2, 2, figsize=(15, 9), sharey=True)
for ax, (name, _) in zip(axes.flatten(), RUSSELL_BANDS.items()):
    sub     = overlap_df[overlap_df['Index'] == name]
    mean_ov = sub['Overlap_Pct'].mean()
    ax.bar(sub['Year'], sub['Overlap_Pct'], color='steelblue', edgecolor='white', zorder=3)
    ax.axhline(100,     color='green',  lw=0.8, linestyle='--', label='Perfect (100%)', zorder=2)
    ax.axhline(mean_ov, color='orange', lw=1.2, linestyle=':',
               label=f'Mean = {mean_ov:.1f}%', zorder=2)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel('Year')
    ax.set_ylabel('Membership Overlap (%)')
    ax.set_ylim(0, 112)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(fontsize=8)

plt.suptitle(
    'Russell Index Membership Overlap: Predicted vs Actual\n'
    '(ARIMA(10,1,10), 60-day horizon)',
    fontsize=13
)
plt.tight_layout()
plt.show()

### 5.5 Add/Remove Prediction — Russell 2000

The hardest and most practically relevant question: can the model identify which stocks will **enter or exit** the Russell 2000 at this year's reconstitution?

**Setup:** prior-year rank date market caps serve as the pre-rebalance membership proxy. A stock is an "add" if it was outside R2000 a year ago and inside at the 60-day target; a "remove" if the reverse.

**Why recall will be structurally low regardless of order (p, q):**  
The stocks that actually cross the R2000 boundary are those with the most *unusual* market cap trajectories — by definition outliers from their own recent trend. A univariate ARIMA model extrapolates recent trend and mean-reversion; it correctly keeps stable stocks where they are but systematically misses the outlier moves that drive actual adds and deletes. Improving p and q order cannot fix this — it is a fundamental limitation of the univariate time-series approach. Cross-sectional signals (earnings surprise, sector momentum, analyst revisions) would be needed.

In [ ]:
rebal_rows = []
for year, cmp in year_results.items():
    if cmp['prev_rank'].isna().all():
        continue
    for name, (lo, hi) in RUSSELL_BANDS.items():
        prev_in   = in_band(cmp['prev_rank'],   lo, hi)
        pred_in   = in_band(cmp['pred_rank'],   lo, hi)
        actual_in = in_band(cmp['actual_rank'], lo, hi)

        actual_adds    = (~prev_in) & actual_in
        actual_removes = prev_in   & (~actual_in)
        pred_adds      = (~prev_in) & pred_in
        pred_removes   = prev_in   & (~pred_in)

        tp_adds    = (pred_adds    & actual_adds).sum()
        tp_removes = (pred_removes & actual_removes).sum()
        n_aa = actual_adds.sum();    n_pa = pred_adds.sum()
        n_ar = actual_removes.sum(); n_pr = pred_removes.sum()

        rebal_rows.append({
            'Year':             year,
            'Index':            name,
            'Actual Adds':      n_aa,
            'Add_Precision':    tp_adds    / n_pa if n_pa > 0 else np.nan,
            'Add_Recall':       tp_adds    / n_aa if n_aa > 0 else np.nan,
            'Actual Removes':   n_ar,
            'Remove_Precision': tp_removes / n_pr if n_pr > 0 else np.nan,
            'Remove_Recall':    tp_removes / n_ar if n_ar > 0 else np.nan,
        })

rebal_df = pd.DataFrame(rebal_rows)
r2k = rebal_df[rebal_df['Index'] == 'Russell 2000'].set_index('Year')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (metric_p, metric_r, label) in zip(axes, [
    ('Add_Precision',    'Add_Recall',    'Adds'),
    ('Remove_Precision', 'Remove_Recall', 'Removes'),
]):
    w = 0.35
    x = np.arange(len(r2k))
    ax.bar(x - w/2, r2k[metric_p] * 100, w, label='Precision', color='steelblue',  alpha=0.85, zorder=3)
    ax.bar(x + w/2, r2k[metric_r] * 100, w, label='Recall',    color='darkorange', alpha=0.85, zorder=3)
    ax.axhline(r2k[metric_p].mean() * 100, color='steelblue',  lw=1.3, linestyle=':',
               label=f'Avg Precision = {r2k[metric_p].mean():.0%}', zorder=2)
    ax.axhline(r2k[metric_r].mean() * 100, color='darkorange', lw=1.3, linestyle=':',
               label=f'Avg Recall = {r2k[metric_r].mean():.0%}', zorder=2)
    ax.set_xticks(x)
    ax.set_xticklabels(r2k.index, rotation=45)
    ax.set_ylabel('%', fontsize=11)
    ax.set_ylim(0, 112)
    ax.set_title(f'Russell 2000 {label}: Precision & Recall by Year', fontsize=11)
    ax.legend(fontsize=8)

plt.suptitle(
    'Russell 2000 Membership Change Prediction  (ARIMA(10,1,10), 60-day horizon)\n'
    'Prior membership proxy = previous year\'s rank date market caps',
    fontsize=12
)
plt.tight_layout()
plt.show()

print('Russell 2000 Add/Remove Summary (mean ± std across years):')
print(r2k[['Add_Precision', 'Add_Recall', 'Remove_Precision', 'Remove_Recall']]
      .agg(['mean', 'std'])
      .applymap(lambda v: f'{v:.1%}'))

### 5.6 Pooled Error Distribution

Aggregating all stock-years gives a single view of the model's accuracy distribution across the full study period.

The **CDF** shows what fraction of stock-years had rank error at or below each threshold. Reference lines at ±50, ±100, ±250, ±500 positions map to practical relevance:
- **≤50**: stock is essentially in the right neighbourhood; minimal index impact
- **≤100**: within 3% of the rank range — acceptable for coarse membership prediction
- **>500**: complete mislocation — model put the stock in the wrong segment of the size spectrum

The **ρ histogram** shows whether performance is consistent across years or driven by a handful of easy years. A tight distribution around a high mean would be reassuring; wide spread or a long left tail suggests the model fails badly in specific market conditions.

In [ ]:
all_errors = pd.concat([cmp['rank_error'] for cmp in year_results.values()])
all_pred   = pd.concat([cmp['predicted']  for cmp in year_results.values()])
all_actual = pd.concat([cmp['actual']     for cmp in year_results.values()])

pooled_rho, _ = spearmanr(all_pred, all_actual)
pooled_mae    = all_errors.mean()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# — CDF —
ax = axes[0]
sorted_err = np.sort(all_errors.values)
cdf = np.arange(1, len(sorted_err) + 1) / len(sorted_err) * 100
ax.plot(sorted_err, cdf, color='steelblue', lw=2)
for thresh, ypos in [(50, 12), (100, 20), (250, 30), (500, 40)]:
    pct = (all_errors <= thresh).mean() * 100
    ax.axvline(thresh, color='gray', linestyle='--', lw=0.8)
    ax.text(thresh + 10, ypos, f'{pct:.0f}% ≤ {thresh}', fontsize=8.5, color='dimgray')
ax.set_xlabel('Absolute Rank Error', fontsize=11)
ax.set_ylabel('Cumulative % of Stock-Years', fontsize=11)
ax.set_title(
    f'CDF of Rank Errors — All Years Pooled\n'
    f'n = {len(all_errors):,} stock-years  |  MAE = {pooled_mae:.0f}  |  ρ = {pooled_rho:.4f}',
    fontsize=11
)
ax.set_ylim(0, 100)

# — Per-year ρ distribution —
ax = axes[1]
rho_vals = [spearmanr(cmp['predicted'], cmp['actual'])[0] for cmp in year_results.values()]
ax.hist(rho_vals, bins=10, color='steelblue', edgecolor='white', zorder=3)
ax.axvline(np.mean(rho_vals), color='darkorange', lw=1.5, linestyle='--',
           label=f'Mean ρ = {np.mean(rho_vals):.3f}')
ax.set_xlabel('Spearman ρ', fontsize=11)
ax.set_ylabel('Number of Years', fontsize=11)
ax.set_title('Distribution of Per-Year Spearman ρ', fontsize=11)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'Pooled Spearman ρ  : {pooled_rho:.4f}')
print(f'Pooled MAE rank    : {pooled_mae:.1f}')
print(f'Total stock-years  : {len(all_errors):,}')